# STAF — Colab Launcher

This notebook is just a thin wrapper. All the actual model code lives in the repo:

- **Training loop** → [`train.py`](https://github.com/RAJ-VARUN13/staf-forensics-multimodal/blob/main/train.py)
- **Evaluation** → [`evaluate.py`](https://github.com/RAJ-VARUN13/staf-forensics-multimodal/blob/main/evaluate.py)
- **Models / datasets / preprocessing** → `staf/`

You can safely **Run All** (`Ctrl+F9`).  
Re-running is safe — clone and unzip steps are idempotent.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

REPO = 'staf-forensics-multimodal'
if not os.path.isdir(f'/content/{REPO}'):
    !git clone https://github.com/RAJ-VARUN13/{REPO}.git
else:
    print(f'{REPO}/ already exists, pulling latest...')
    !git -C /content/{REPO} pull --ff-only

%cd /content/{REPO}
!pip install -q -r requirements.txt

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        'No GPU detected. Go to Runtime > Change runtime type and pick a GPU.'
    )

print(f'GPU ready: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# point this at wherever you uploaded the zip on Drive
ZIP_PATH = '/content/drive/MyDrive/FakeAVCeleb_v1.2.zip'
EXTRACT_TO = '/content'

import os
if not os.path.isfile(ZIP_PATH):
    raise FileNotFoundError(
        f'Dataset zip not found at {ZIP_PATH}\n'
        f'Upload FakeAVCeleb_v1.2.zip to your Google Drive and fix the path above.'
    )

# -n = never overwrite (safe for re-runs)
!unzip -qn "{ZIP_PATH}" -d {EXTRACT_TO}
print('Dataset ready at', os.path.join(EXTRACT_TO, 'FakeAVCeleb'))

In [ ]:
CONFIG = 'staf/configs/colab.yaml'

!python -m staf.preprocessing.extract_frames  --config {CONFIG}
!python -m staf.preprocessing.detect_faces    --config {CONFIG}
!python -m staf.preprocessing.crop_faces      --config {CONFIG}
!python -m staf.preprocessing.extract_audio   --config {CONFIG}
!python -m staf.preprocessing.sync_modalities --config {CONFIG}

In [ ]:
!python train.py --config {CONFIG} --no_wandb

In [ ]:
import glob

checkpoints = sorted(glob.glob('/content/results/*/checkpoints/best_model.pt'))
if not checkpoints:
    raise FileNotFoundError(
        'No best_model.pt found under /content/results/. '
        'Did training finish or get interrupted?'
    )

ckpt = checkpoints[-1]   # latest run
print(f'Evaluating: {ckpt}')
!python evaluate.py --checkpoint "{ckpt}" --split test

In [ ]:
import os, shutil

DRIVE_DEST = '/content/drive/MyDrive/STAF_experiments'
os.makedirs(DRIVE_DEST, exist_ok=True)

!cp -ru /content/results/* "{DRIVE_DEST}/"
print(f'Results backed up to {DRIVE_DEST}')